In [76]:
import numpy as np

from tensorflow.keras import layers

from tensorflow.keras.models import Sequential
 
from tensorflow.keras.layers import Embedding

from tensorflow.keras.layers import SimpleRNN,LSTM

from tensorflow.keras.layers import Dense

from tensorflow.keras.layers import TimeDistributed
 
from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.preprocessing.sequence import pad_sequences

import tensorflow as tf
 

In [77]:
# Define texts and labels (from WC119BP8nKKu)
texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever"
]

In [78]:
labels=np.array([1,1,1,1,0,0,0,0])

In [79]:
vocab_size=1000
tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences=tokenizer.texts_to_sequences(texts)
max_length = int(np.mean([len(seq) for seq in sequences]))
X=pad_sequences(sequences,maxlen=max_length,padding="post")

print("WOrd Index:")

print(tokenizer.word_index)
print("\nInput sequences:")
print(X)

print(max_length)

WOrd Index:
{'<OOV>': 1, 'this': 2, 'i': 3, 'movie': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'love': 9, 'is': 10, 'amazing': 11, 'good': 12, 'excellent': 13, 'hate': 14, 'terrible': 15, 'boring': 16, 'worst': 17, 'ever': 18}

Input sequences:
[[ 9  2  4]
 [ 5 10 11]
 [ 6 12  7]
 [13  8  0]
 [14  2  4]
 [15  5  0]
 [ 6 16  8]
 [17  7 18]]
3


## TOKEN EMBEDDING 
 we are doing it in self.token_embedding.

 what does it do: Converts each word (token) index into a dense vector of numbers.

 why it is needed: The model can't work with raw integers.

 what if it is removed: The model has no way to represent words.

## POSITIONAL EMBEDDING 

 we are doing it in self.position_embedding

 what does it do: Adds a vector for each position 

 why it is needed: Positional embeddings tell the model where each word is in the sentence.

 what if it is removed: Word order would be completely lost.

In [80]:
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb


# 3 MUTLI HEAD ATTENTION

self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

What it does: it computes how much attention to pay to every other word. 

Why it's needed: For "I love this movie", it learns that "love" is strongly related to the overall sentiment.

If removed: The model loses its ability to relate words to each other contextually

# 4 FEED FORWORD NN

ffn_output = self.ffn(out1)

What it does: A small 2-layer neural network applied to each position's output from the attention layer.

Why it's needed: Allowing the model to learn more complex feature representations at each position.

If removed: The model would be less expressive. It could still learn some patterns but we cant get accurate answers

# 5 ADD & NORMALIZE

out1 = self.layernorm1(inputs + attention_output)

What it does: Takes the output of the Transformer block, and averages across all 6 positions to get a single vector of shape (batch, 16).

Why it's needed: The final Dense layer needs a fixed-size input. This layer collapses the sequence dimension by averaging, giving a single summary vector for the whole sentence.

If removed: You can't directly connect the Transformer's 3D output (batch, 6, 16) to a Dense layer that expects 2D input (batch, features). The model would will crash .

In [81]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
 
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

# OUTPUT LAYER

x = layers.GlobalAveragePooling1D()(x)

What it does:What it does:After processing, each word has its own set of numbers. This just takes all the words and averages them into one single summary.

Why it's needed: The final layer needs one answer, not 6 separate ones. This is how we combine them.

If removed:The model crashes — you're trying to feed 6 things into a layer that only accepts 1.



In [ ]:
# build the model
embed_dim = 16
num_heads = 2
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(1, activation="sigmoid")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [83]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_15 (InputLayer)     │ (None, 3)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_12 │ (None, 3, 16)          │        16,048 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 3, 16)          │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,361 (75.63 KB)

 Trainable params: 19,361 (75.63 KB)

 Non-trainable params: 0 (0.00 B)

In [84]:
model.fit(
    X,
    labels,
    epochs=30,
    batch_size=2,
    verbose=1
)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.6250 - loss: 0.7366
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5000 - loss: 0.6347    
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8750 - loss: 0.5665 
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.5200 
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8750 - loss: 0.4880
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8750 - loss: 0.4167
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.3739
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.3277 
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.2822 
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.2422 
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.2139 
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1815

In [85]:
test_sentences = [
    "I love the film",
    "This movie was awful"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
I love the film -> 0.9074378
Prediction: Positive
This movie was awful -> 0.20613834
Prediction: Negative


```mermaid
flowchart TD
    A["Raw text\n'I love this movie'"]
    B["Token IDs\n[3, 9, 2, 4, 0, 0]"]
    C["Token + Position Embedding\n6 words × 16 features"]
    D["Multi-Head Attention"]
    E["Add & Normalize"]
    F["Feed-Forward Network"]
    G["Add & Normalize"]
    H["GlobalAveragePooling1D\n6×16 → 1×16"]
    I["Dense + relu\n→ score 0 to 1"]
    J{"Score > 0.5?"}
    K["Positive ✓"]
    L["Negative ✗"]

    A --> B --> C --> D --> E --> F --> G --> H --> I --> J
    J -->|Yes| K
    J -->|No| L
```

TASK 4

1. mehatab is from raichur and it is bad
2. rahul is from dharwad and it is good

THE neutral texts will be need to be trained more to get it correctly and we need 1 more layer


task 6

HEADS 8

EMBEDDING DIMENSION 32 

WILL BE GOOD
